# 載入函式庫與資料集

In [1]:
import glob
import json
import os
import shutil

import numpy as np
import optuna
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical


/Users/lin/Downloads/mlp-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 載入已經預處理好的資料
> 上述資料集整理處理完，往後即能直接載入資料集，不需每次都要重新處理原始資料

In [2]:
np.random.seed(627)  # 指定亂數種子：只要不更改，每次運作時即可保持再現性
# 載入Titanic的訓練和測試資料集
df_train = pd.read_csv("./titanic_train.csv", dtype='float')
df_test = pd.read_csv("./titanic_test.csv", dtype='float')
dataset_train = df_train.values.copy()
dataset_test = df_test.values.copy()

In [3]:

# 分割成特徵資料和標籤資料
X_train = dataset_train[:, 0:9]
Y_train = dataset_train[:, 9]
X_test = dataset_test[:, 0:9]
Y_test = dataset_test[:, 9]

# 特徵標準化
X_train -= X_train.mean(axis=0)
X_train /= X_train.std(axis=0)
X_test -= X_test.mean(axis=0)
X_test /= X_test.std(axis=0)

# 建立模型
> 若已載入先前訓練過的模型，則此處不要執行。

In [4]:
# 使用 Optuna 快速模式定義模型與搜尋目標
# 快速模式：搜尋空間收窄，但每次 trial 仍儲存 .keras，最後只保留最佳模型。
SEARCH_LOG = "Ch6_2_optuna_search.json"
FINAL_MODEL = "413570012_0501.keras"
KEEP_MODEL = "413570012_20260424.keras"
N_TRIALS = 1000
BASE_SEED = 20260501

trial_records = []
best_score = -np.inf
best_record = None
best_file = None
best_history = None


def build_model_from_params(params):
    model = Sequential()
    activation = params["activation"]
    model.add(Dense(params["units_1"], input_dim=X_train.shape[1], activation=activation))

    if params["n_layers"] == 2:
        model.add(Dense(params["units_2"], activation=activation))

    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model


def build_model(trial):
    # 搜尋空間集中在目前最高分附近；保留少量小雙層模型作探索。
    n_layers = trial.suggest_categorical("n_layers", [1, 1, 1, 2])
    params = {
        "n_layers": n_layers,
        "activation": trial.suggest_categorical("activation", ["softplus", "softplus", "softplus", "relu"]),
        "units_1": trial.suggest_categorical("units_1", [3, 3, 3, 4, 5]),
    }
    if n_layers == 2:
        params["units_2"] = trial.suggest_categorical("units_2", [1, 2])
    return build_model_from_params(params)


def score_model(model):
    _, accuracy_train = model.evaluate(X_train, Y_train, verbose=0)
    _, accuracy_test = model.evaluate(X_test, Y_test, verbose=0)
    score = (accuracy_test - np.abs(accuracy_train - accuracy_test)) * 100 + 10
    return float(score), float(accuracy_train), float(accuracy_test)


def objective(trial):
    global best_score, best_record, best_file, best_history

    seed = BASE_SEED + trial.number
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    model = build_model(trial)
    validation_split = trial.suggest_categorical(
        "validation_split",
        [0.0, 0.005, 0.01, 0.0125, 0.015, 0.0175, 0.02, 0.025, 0.035, 0.05],
    )
    epochs = trial.suggest_int("epochs", 60, 82)
    batch_size = trial.suggest_categorical(
        "batch_size", [104, 112, 120, 128, 136, 144, 152, 160, 176, 192, 208]
    )

    history = model.fit(
        X_train,
        Y_train,
        validation_split=validation_split,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
    )

    score, accuracy_train, accuracy_test = score_model(model)
    trial_file = f"score_{score:.6f}_optuna_{trial.number:04d}.keras"
    model.save(trial_file)

    record = {
        "trial": trial.number,
        "score": score,
        "accuracy_train": accuracy_train,
        "accuracy_test": accuracy_test,
        "train_correct": int(round(accuracy_train * len(Y_train))),
        "test_correct": int(round(accuracy_test * len(Y_test))),
        "seed": seed,
        "file": trial_file,
        **trial.params,
    }
    trial_records.append(record)
    trial.set_user_attr("file", trial_file)
    trial.set_user_attr("score_record", record)

    if score > best_score:
        best_score = score
        best_record = record
        best_file = trial_file
        best_history = history
        print("NEW_BEST", json.dumps(record, ensure_ascii=False))

    if trial.number % 25 == 0:
        with open(SEARCH_LOG, "w") as f:
            json.dump(sorted(trial_records, key=lambda r: r["score"], reverse=True), f, indent=2)

    return score


# 訓練神經網路

In [5]:
# 建立 Optuna study，並把目前最佳附近的組合先排進佇列
for path in glob.glob("score_*.keras"):
    os.remove(path)

if os.path.exists(FINAL_MODEL):
    current_model = load_model(FINAL_MODEL)
    best_score, accuracy_train, accuracy_test = score_model(current_model)
    best_file = "score_current_best.keras"
    current_model.save(best_file)
    best_record = {
        "trial": "current",
        "score": best_score,
        "accuracy_train": accuracy_train,
        "accuracy_test": accuracy_test,
        "train_correct": int(round(accuracy_train * len(Y_train))),
        "test_correct": int(round(accuracy_test * len(Y_test))),
        "seed": None,
        "file": best_file,
        "n_layers": 1,
        "activation": "softplus",
        "units_1": 3,
        "epochs": 65,
        "batch_size": 128,
        "validation_split": 0.015,
    }
    trial_records.append(best_record)

sampler = optuna.samplers.TPESampler(seed=BASE_SEED, n_startup_trials=80)
study = optuna.create_study(direction="maximize", sampler=sampler)

# 先測目前最佳附近，再交給 TPE sampler 自動探索。
for _ in range(12):
    for params in [
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 65, "batch_size": 128, "validation_split": 0.015},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 64, "batch_size": 128, "validation_split": 0.015},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 66, "batch_size": 128, "validation_split": 0.015},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 65, "batch_size": 120, "validation_split": 0.015},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 65, "batch_size": 136, "validation_split": 0.015},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 65, "batch_size": 128, "validation_split": 0.0125},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 65, "batch_size": 128, "validation_split": 0.0175},
        {"n_layers": 1, "activation": "softplus", "units_1": 3, "epochs": 79, "batch_size": 128, "validation_split": 0.0175},
        {"n_layers": 2, "activation": "relu", "units_1": 4, "units_2": 2, "epochs": 76, "batch_size": 208, "validation_split": 0.025},
    ]:
        study.enqueue_trial(params)


[I 2026-05-01 09:08:51,070] A new study created in memory with name: no-name-3aa14364-9fda-4835-ba2a-b5ca934bd2f4


In [6]:
# 執行 Optuna 快速搜尋
print("Optuna fast tuning ...")
study.optimize(objective, n_trials=N_TRIALS)

trial_records = sorted(trial_records, key=lambda r: r["score"], reverse=True)
with open(SEARCH_LOG, "w") as f:
    json.dump(trial_records, f, indent=2)

best_record = trial_records[0]
best_file = best_record["file"]
print("Best score = {:.9f}".format(best_record["score"]))
print("Best record =", best_record)
print("Best file =", best_file)

model = load_model(best_file)
# history = best_history

# # 讓後面的圖表 cell 在最佳 trial 沒有 validation history 時仍可執行。
# if history is None:
#     class EmptyHistory:
#         history = {
#             "loss": [np.nan],
#             "val_loss": [np.nan],
#             "accuracy": [np.nan],
#             "val_accuracy": [np.nan],
#         }
#     history = EmptyHistory()
# else:
#     n = len(history.history.get("loss", []))
#     if "val_loss" not in history.history:
#         history.history["val_loss"] = [np.nan] * n
#     if "val_accuracy" not in history.history:
#         history.history["val_accuracy"] = [np.nan] * n


/Users/lin/Downloads/mlp-prediction/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Optuna fast tuning ...


[I 2026-05-01 09:08:55,610] Trial 0 finished with value: 83.47131371498108 and parameters: {'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.015, 'epochs': 65, 'batch_size': 128}. Best is trial 0 with value: 83.47131371498108.


[I 2026-05-01 09:09:00,695] Trial 1 finished with value: 78.33567023277283 and parameters: {'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.015, 'epochs': 64, 'batch_size': 128}. Best is trial 0 with value: 83.47131371498108.


[I 2026-05-01 09:09:05,815] Trial 2 finished with value: 75.56361317634583 and parameters: {'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.015, 'epochs': 66, 'batch_size': 128}. Best is trial 0 with value: 83.47131371498108.


...

[I 2026-05-01 11:16:43,568] Trial 996 finished with value: 75.92063903808594 and parameters: {'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.01, 'epochs': 63, 'batch_size': 144}. Best is trial 450 with value: 89.55955862998962.


[I 2026-05-01 11:16:56,328] Trial 997 finished with value: 79.19395327568054 and parameters: {'n_layers': 2, 'activation': 'relu', 'units_1': 3, 'units_2': 1, 'validation_split': 0.015, 'epochs': 78, 'batch_size': 104}. Best is trial 450 with value: 89.55955862998962.


[I 2026-05-01 11:17:08,869] Trial 998 finished with value: 77.99501180648804 and parameters: {'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.015, 'epochs': 62, 'batch_size': 120}. Best is trial 450 with value: 89.55955862998962.


[I 2026-05-01 11:17:18,640] Trial 999 finished with value: 83.83691668510437 and parameters: {'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.0, 'epochs': 65, 'batch_size': 128}. Best is trial 450 with value: 89.55955862998962.


Best score = 89.559558630
Best record = {'trial': 450, 'score': 89.55955862998962, 'accuracy_train': 0.7960711121559143, 'accuracy_test': 0.7958333492279053, 'train_correct': 851, 'test_correct': 191, 'seed': 20260951, 'file': 'score_89.559559_optuna_0450.keras', 'n_layers': 1, 'activation': 'softplus', 'units_1': 3, 'validation_split': 0.015, 'epochs': 68, 'batch_size': 120}
Best file = score_89.559559_optuna_0450.keras


In [7]:
# # 顯示圖表來分析模型的訓練過程
# import matplotlib.pyplot as plt
# # 顯示訓練和驗證損失
# loss = history.history["loss"]
# epochs = range(1, len(loss)+1)
# val_loss = history.history["val_loss"]
# plt.plot(epochs, loss, "b-", label="Training Loss")
# plt.plot(epochs, val_loss, "r--", label="Validation Loss")
# plt.title("Training and Validation Loss")
# plt.xlabel("Epochs")
# plt.ylabel("Loss")
# plt.legend()
# plt.show()
# # 顯示訓練和驗證準確度
# acc = history.history["accuracy"]
# epochs = range(1, len(acc)+1)
# val_acc = history.history["val_accuracy"]
# plt.plot(epochs, acc, "b-", label="Training Acc")
# plt.plot(epochs, val_acc, "r--", label="Validation Acc")
# plt.title("Training and Validation Accuracy")
# plt.xlabel("Epochs")
# plt.ylabel("Accuracy")
# plt.legend()
# plt.show()

In [8]:
# 作業分數評分公式:(測試資料集準確度-|訓練資料準確度-測試資料準確度|)*100+10

loss, accuracy_train = model.evaluate(X_train, Y_train, verbose=0)
print("訓練資料集的準確度 = {:.2f}".format(accuracy_train))
loss, accuracy_test = model.evaluate(X_test, Y_test, verbose=0)
print("測試資料集的準確度 = {:.2f}".format(accuracy_test))

score = (accuracy_test - np.abs(accuracy_train-accuracy_test))*100 + 10
print("作業分數 = {:.0f}".format(score))

訓練資料集的準確度 = 0.80
測試資料集的準確度 = 0.80
作業分數 = 90


In [9]:
# 儲存最佳 Keras 模型，並清理 Optuna trial 暫存模型
print("Saving Model ...")
model.save(FINAL_MODEL) #學號+日期當作主檔名

for path in glob.glob("score_*.keras"):
    os.remove(path)


Saving Model ...
